# Benchmark de `fit`: Bankai vs. LightGBM

Gera uma classificação binária sintética com 1.000.000 de linhas e 500 features. As primeiras 90 features entram com coeficientes não nulos na geração do alvo; as outras 410 são ruído independente.

As células de benchmark cronometram somente `model.fit(X, y)`. Importações, geração dos dados e criação dos estimadores ficam fora do cronômetro. A configuração atual usa importance gain nos dois estimadores, sem calcular importância por permutação.

In [1]:
import os
import platform
import time

import lightgbm as lgb
import numpy as np
from bankai_random_forest import BankaiRandomForestClassifier

print(f'Python: {platform.python_version()}')
print(f'NumPy: {np.__version__}')
print(f'LightGBM: {lgb.__version__}')
print(f'CPUs lógicas visíveis: {os.cpu_count()}')

Python: 3.11.11
NumPy: 2.4.6
LightGBM: 4.7.0
CPUs lógicas visíveis: 8


In [2]:
N_SAMPLES = 100_000
N_FEATURES = 500
N_RELEVANT = 90
SEED = 1729

rng = np.random.default_rng(SEED)
# Float32 mantém a matriz em cerca de 200 MB e é aceito pelos dois estimadores.
X = rng.standard_normal((N_SAMPLES, N_FEATURES), dtype=np.float32)

# Cada uma das 90 primeiras features recebe um coeficiente não nulo.
coefficients = rng.choice(np.array([-1.0, 1.0], dtype=np.float32), size=N_RELEVANT)
coefficients *= rng.uniform(0.5, 1.5, size=N_RELEVANT).astype(np.float32)
signal = X[:, :N_RELEVANT] @ coefficients
signal /= np.sqrt(np.sum(coefficients ** 2))
probability = 1.0 / (1.0 + np.exp(-signal))
y = rng.binomial(1, probability).astype(np.uint8)

assert X.shape == (100_000, 500)
assert np.count_nonzero(coefficients) == 90
print(f'X: {X.shape}, dtype={X.dtype}, {X.nbytes / 1e6:.0f} MB')
print(f'y: {y.shape}, classes={np.unique(y, return_counts=True)[1].tolist()}')
print(f'Features relevantes: colunas 0–89 ({np.count_nonzero(coefficients)} coeficientes não nulos)')
print('Features de ruído: colunas 90–499 (410)')

X: (100000, 500), dtype=float32, 200 MB
y: (100000,), classes=[49989, 50011]
Features relevantes: colunas 0–89 (90 coeficientes não nulos)
Features de ruído: colunas 90–499 (410)


## Configuração comum

Os limites de bins, árvores, profundidade, folhas, amostras por folha, features e bagging são aproximados entre os algoritmos. `n_jobs=-1` usa todos os CPUs lógicos visíveis. O Bankai usa uma amostra de 200.000 linhas para os cortes; LightGBM está configurado para 1.000.000 em `subsample_for_bin`. Os critérios de split também diferem, então os tempos não representam trabalho matematicamente idêntico.

In [3]:
N_JOBS = -1
N_ESTIMATORS = 40
MAX_BINS = 63
MAX_DEPTH = 20
MAX_LEAVES = 511
MIN_SAMPLES_LEAF = 5
SAMPLE_FRACTION = 0.8
BANKAI_BINNING_STRATEGY = 'sampled_select'
BANKAI_BIN_SAMPLE_SIZE = 200_000

## Bankai: medir somente `fit`

Este teste usa `sampled_select`: o Bankai seleciona os cortes em uma amostra de até `BANKAI_BIN_SAMPLE_SIZE` linhas. Ajuste `BANKAI_BINNING_STRATEGY` e `BANKAI_BIN_SAMPLE_SIZE` na célula de configuração para comparar as estratégias. A medição continua cobrindo somente o `fit`; a configuração atual usa importance gain.

In [8]:
bankai = BankaiRandomForestClassifier(
    n_estimators=N_ESTIMATORS,
    max_bins=MAX_BINS,
    max_depth=MAX_DEPTH,
    max_leaf_nodes=MAX_LEAVES,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    max_features=None,
    max_samples=SAMPLE_FRACTION,
    n_jobs=N_JOBS,
    random_state=42,
    importance_type='permutation',
    #binning_strategy=BANKAI_BINNING_STRATEGY,
    bin_sample_size=BANKAI_BIN_SAMPLE_SIZE
)

start = time.perf_counter()
bankai.fit(X, y)
bankai_fit_seconds = time.perf_counter() - start
print(f'Bankai fit: {bankai_fit_seconds:.3f} s')

Bankai fit: 12.122 s


## LightGBM: medir somente `fit`

LightGBM usa `boosting_type='rf'`, bagging de 80% e todas as features. Nenhuma importância por permutação é calculada nesta célula.

In [5]:
lightgbm = lgb.LGBMClassifier(
    boosting_type='rf',
    n_estimators=N_ESTIMATORS,
    max_bin=MAX_BINS,
    max_depth=MAX_DEPTH,
    num_leaves=MAX_LEAVES,
    min_child_samples=MIN_SAMPLES_LEAF,
    bagging_freq=1,
    bagging_fraction=SAMPLE_FRACTION,
    feature_fraction=1.0,
    n_jobs=N_JOBS,
    random_state=42,
    importance_type='gain',
    verbosity=-1,
    subsample_for_bin=BANKAI_BIN_SAMPLE_SIZE
)

start = time.perf_counter()
lightgbm.fit(X, y)
lightgbm_fit_seconds = time.perf_counter() - start
print(f'LightGBM fit: {lightgbm_fit_seconds:.3f} s')

LightGBM fit: 13.086 s


## Comparação dos tempos

Esta célula apenas exibe os tempos já medidos; não executa `fit`, predição ou cálculo de métricas. Para reduzir ruído, reinicie o kernel e repita as duas células de benchmark em ordem alternada.

In [6]:
faster = 'Bankai' if bankai_fit_seconds < lightgbm_fit_seconds else 'LightGBM'
ratio = max(bankai_fit_seconds, lightgbm_fit_seconds) / min(bankai_fit_seconds, lightgbm_fit_seconds)
print(f'Bankai:   {bankai_fit_seconds:.3f} s')
print(f'LightGBM: {lightgbm_fit_seconds:.3f} s')
print(f'{faster} foi {ratio:.2f}x mais rápido neste fit.')

Bankai:   12.743 s
LightGBM: 13.086 s
Bankai foi 1.03x mais rápido neste fit.


### Limites da comparação

A medição inclui conversão e pré-processamento executados internamente por cada biblioteca. A geração do dataset e a construção dos estimadores ficam fora. A configuração atual usa importance gain nos dois estimadores, sem permutação. Para testar permutation importance no Bankai, altere `importance_type`; esse custo será incluído no tempo de `fit` e deixará de isolar a comparação das estratégias de discretização.